# PulseML — Exploratory Data Analysis

This notebook walks through the synthetic ICU vital-sign dataset:
- Dataset size and schema
- Distribution of each vital sign
- Deteriorating vs. stable patient trajectories
- Missingness analysis
- Feature correlation heatmap
- Class imbalance in the label

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

# Load raw vitals
raw = pd.read_parquet('../data/raw/vitals.parquet')
print(f'Shape: {raw.shape}')
raw.head()

In [ ]:
# Patient-level summary
patient_summary = raw.groupby('patient_id').agg(
    n_readings=('timestamp_minutes', 'count'),
    stay_hours=('hours_in_icu', 'max'),
    deteriorated=('label', 'max'),
).reset_index()

print(f"Total patients: {len(patient_summary)}")
print(f"Deterioration rate: {patient_summary['deteriorated'].mean():.1%}")
patient_summary.describe()

In [ ]:
# Vital sign distributions — stable vs deteriorating patients
vitals = ['hr', 'sbp', 'dbp', 'spo2', 'rr', 'temp']
det_ids   = set(patient_summary[patient_summary['deteriorated'] == 1]['patient_id'])
stable_ids = set(patient_summary[patient_summary['deteriorated'] == 0]['patient_id'])

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, v in zip(axes.flat, vitals):
    det_vals    = raw[raw['patient_id'].isin(det_ids)][v].dropna()
    stable_vals = raw[raw['patient_id'].isin(stable_ids)][v].dropna()
    ax.hist(stable_vals, bins=40, alpha=0.6, label='Stable', color='steelblue', density=True)
    ax.hist(det_vals,    bins=40, alpha=0.6, label='Deteriorating', color='tomato', density=True)
    ax.set_title(v.upper())
    ax.legend(fontsize=8)
plt.suptitle('Vital Sign Distributions: Stable vs Deteriorating', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Example patient trajectories — one deteriorating, one stable
det_pid    = list(det_ids)[0]
stable_pid = list(stable_ids)[0]

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
for ax, v in zip(axes.flat, vitals):
    for pid, label, color in [(det_pid, 'Deteriorating', 'tomato'),
                               (stable_pid, 'Stable', 'steelblue')]:
        pat = raw[raw['patient_id'] == pid].sort_values('hours_in_icu')
        ax.plot(pat['hours_in_icu'], pat[v], alpha=0.8, label=label, color=color)
    ax.set_title(v.upper()); ax.set_xlabel('Hours in ICU')
    ax.legend(fontsize=8)
plt.suptitle('Example Patient Trajectories', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Missingness heatmap
miss = raw[vitals].isna().mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 3))
miss.plot.bar(ax=ax, color='steelblue', alpha=0.8)
ax.set_ylabel('Missing rate')
ax.set_title('Per-vital Missing Rate')
plt.tight_layout()
plt.show()
print(miss)

In [ ]:
# Feature correlation (on computed features)
from src.pipeline.features import compute_window_features, get_feature_columns

# Sample 500 patients for speed
sample_pids = np.random.choice(raw['patient_id'].unique(), size=500, replace=False)
sample_raw  = raw[raw['patient_id'].isin(sample_pids)]
features    = compute_window_features(sample_raw, window_hours=6)
feat_cols   = get_feature_columns(features)

corr = features[feat_cols].corr()
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.3, square=True, cbar_kws={'shrink': 0.6})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Label distribution
label_counts = features['label'].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
label_counts.plot.bar(ax=ax, color=['steelblue', 'tomato'], rot=0)
ax.set_xticklabels(['Stable (0)', 'Deteriorating (1)'])
ax.set_title('Label Distribution in Feature Dataset')
ax.set_ylabel('Count')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{int(bar.get_height()):,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()
print(f'Positive rate: {features["label"].mean():.2%}')